# Docling OCR Testing

**objective:**
testing docling's capabilities to extract text from images and pdfs in one shot, meaning without having to do any cropping nor background removal, on two types of docs: CINs, and Papyrus.

## Setup

In [ ]:
import io

to use docling with tesseract engine, you should:
- install tesseract and tesseract-lang via homebrew
- manually moving files (fra.traineddata and ara.traineddata) from tesseract-lang to tesseract
- set the TESSDATA_PREFIX environment variable to the path of the tesseract share folder


commands on mac:
```bash
brew install tesseract
brew install tesseract-lang 
brew info tesseract
brew info tesseract-lang
tesseract --list-langs  
cp /opt/homebrew/Cellar/tesseract-lang/4.1.0/share/tessdata/fra.traineddata /opt/homebrew/Cellar/tesseract/5.5.2/share/tessdata 
cp /opt/homebrew/Cellar/tesseract-lang/4.1.0/share/tessdata/ara.traineddata /opt/homebrew/Cellar/tesseract/5.5.2/share/tessdata 
```

In [ ]:
import os
import time
from pathlib import Path

import fitz
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TesseractOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from PIL import Image

os.environ["TESSDATA_PREFIX"] = "/opt/homebrew/Cellar/tesseract/5.5.2/share/tessdata/"

## Helpers

In [ ]:
# Define CIN image files
data_dir = Path("../data")
cin_files = [
    data_dir / "trash_cin.jpg",
    data_dir / "trash_cin_1.jpg",
    data_dir / "trash_cin_1.pdf",
    data_dir / "trash_cin_2.jpg",
    data_dir / "trash_cin_2.pdf",
]

print(f"Found {len(cin_files)} CIN files to process")
for f in cin_files:
    print(f"  - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
def plot_doc(path):
    """Plot image or PDF file based on file suffix"""
    file_extension = path.suffix.lower()
    if file_extension in [".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".gif", ".webp"]:
        # Display image
        try:
            img = mpimg.imread(str(path))
            plt.figure(figsize=(10, 8))
            plt.imshow(img)
            plt.title(f"Image: {path.name}")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
            return True
        except Exception as e:
            print(f"⚠️  Could not display image: {e}")
            return False

    elif file_extension == ".pdf":
        # Display PDF (first page)
        try:
            pdf_document = fitz.open(str(path))
            if pdf_document.page_count > 0:
                page = pdf_document.load_page(0)
                pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for better quality

                # Convert to image
                img_data = pix.tobytes("png")
                img = Image.open(io.BytesIO(img_data))

                plt.figure(figsize=(10, 8))
                plt.imshow(img)
                plt.title(f"PDF (Page 1 of {pdf_document.page_count}): {path.name}")
                plt.axis("off")
                plt.tight_layout()
                plt.show()

                print(f"📄 PDF has {pdf_document.page_count} page(s)")
                pdf_document.close()
                return True
            else:
                print("⚠️  PDF has no pages")
                pdf_document.close()
                return False
        except Exception as e:
            print(f"⚠️  Could not display PDF: {e}")
            return False

    else:
        print(f"⚠️  Unsupported file type for preview: {file_extension}")
        return False

In [ ]:
def convert(converter, path, show_doc=True):
    """show image, compute conversion or OCR time, outputs result"""
    print(f"================== Processing {path.name}... ==================")
    if show_doc:
        plot_doc(path)
    tic = time.time()
    result = converter.convert(str(path))
    toc = time.time()
    print(f"================== Processing time: ({toc-tic:.2f}s) ==================")
    text_items = result.document.texts
    text_str = "\n".join([text_item.text for text_item in text_items])
    print(f"================== Results: ==================\n {text_str}")

## Docling's Default OCR on CINs

In [ ]:
converter = DocumentConverter()

In [ ]:
convert(converter, cin_files[0])

In [ ]:
convert(converter, cin_files[1])

In [ ]:
convert(converter, cin_files[2])

In [ ]:
convert(converter, cin_files[3])

In [ ]:
convert(converter, cin_files[4])

**Takeways:**

- once onto memory, around 2 seconds to process images and pdfs, colorful images take a bit more 
- decent information extraction conditioned to image quality
- direct extraction from pdf seems to have a slightly better extraction quality, but it can also be an artifact of the pdf-image conversion process
- pdf extraction extracts moreover some arabic characters, it's however rubbish character sequences still

## Docling's default OCR on entire papyrus 

In [ ]:
convert(converter, data_dir / "trash_papyrus_1.pdf", show_doc=False)

In [ ]:
convert(converter, data_dir / "trash_papyrus_2.pdf", show_doc=False)

## Improving on Docling's Default OCR

In [ ]:
ocr_options = TesseractOcrOptions(lang=["ara", "fra"])
pipeline_options = PdfPipelineOptions(ocr_options=ocr_options)
tesseract_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)},
)

In [ ]:
convert(tesseract_converter, cin_files[1])

In [ ]:
convert(tesseract_converter, cin_files[2])

**Takeaways:**
- using tesseract as OCR engine doesn't seem to yield better results than the default engine
- neither on time nor accuracy